In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 04_generate_preferences.ipynb
# Purpose: Generate preference pairs for Direct Preference Optimization (DPO).
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# ============================================================
# Project Paths
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM"
)

print(PROJECT_ROOT)
print(PROJECT_ROOT.exists())

/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM
True


In [ ]:
# ============================================================
# Add Project to Python Path
# ============================================================

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(sys.path[-1])

/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM


In [ ]:
# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

ADAPTER_PATH = PROJECT_ROOT / "models" / "sft_qwen"

PROMPT_PATH = (
    PROJECT_ROOT
    / "prompts"
    / "P01_Zero_Shot_Closed_Coding.txt"
)

TRAIN_SPLIT = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "train.csv"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "dpo"
    / "train.jsonl"
)

MAX_NEW_TOKENS = 250

In [ ]:
# ============================================================
# Install Dependencies
# ============================================================

!pip install -q \
transformers \
peft \
accelerate \
datasets \
sentencepiece

In [ ]:
# ============================================================
# Imports
# ============================================================

import json

import pandas as pd

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from peft import (
    PeftModel,
)

from utils.inference import (
    predict_quote,
    parse_response,
)

In [ ]:
# ============================================================
# Load Prompt
# ============================================================

PROMPT = PROMPT_PATH.read_text(
    encoding="utf-8"
)

print(PROMPT[:500])

You are a software engineering researcher specializing in qualitative analysis of psychological safety in software development teams.

Your task is to perform qualitative coding of the quotes provided below by identifying and classifying psychological safety challenges described in each quote.

The goal is to identify challenges related to psychological safety, such as interpersonal risk-taking, learning anxiety, defensive behaviors, or resistance to change.

---

Context for Classification

Psy


In [ ]:
# ============================================================
# Load Dataset
# ============================================================

import pandas as pd

from sklearn.model_selection import train_test_split

quotes = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "quotes.csv"
)

gold = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "gold_standard.csv"
)

dataset = quotes.merge(
    gold,
    on="quote_id",
)

dataset = dataset.rename(
    columns={
        "category": "gold_category",
    }
)

dataset = dataset[
    [
        "quote_id",
        "quote_text",
        "gold_category",
    ]
]

train_df, _ = train_test_split(
    dataset,
    train_size=0.80,
    shuffle=True,
    random_state=42,
)

train_df = train_df.reset_index(drop=True)

print(train_df.shape)

train_df.head()

(92, 3)


,quote_id,quote_text,gold_category
0,Quote 16,How to manage a failed sub team inside a scrum...,Drawing attention to errors
1,Quote 69,How do I convince my fellow devs to WANT to ad...,Recommending changes
2,Quote 32,It's such a confusing prospect for me as a scr...,Expressing concerns
3,Quote 25,My boss missed two retrospectives because he h...,Expressing concerns
4,Quote 56,when (if ever) is it appropriate for a Scrum M...,Expressing concerns


In [ ]:
quotes = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "quotes.csv"
)

gold = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "gold_standard.csv"
)

dataset = quotes.merge(
    gold,
    on="quote_id",
)

dataset = dataset.rename(
    columns={
        "category": "gold_category"
    }
)

dataset = dataset[
    [
        "quote_id",
        "quote_text",
        "gold_category",
    ]
]

from sklearn.model_selection import train_test_split

train_df, _ = train_test_split(
    dataset,
    train_size=0.80,
    shuffle=True,
    random_state=42,
)

train_df = train_df.reset_index(drop=True)

print(train_df.shape)

(92, 3)


In [ ]:
# ============================================================
# Load SFT Model
# ============================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from peft import (
    PeftModel,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    model,
    ADAPTER_PATH,
)

model.eval()

print("SFT model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

SFT model loaded successfully.


In [ ]:
from tqdm.auto import tqdm

predictions = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df)):

    response = predict_quote(
        model=model,
        tokenizer=tokenizer,
        prompt_template=PROMPT,
        quote_id=row["quote_id"],
        quote_text=row["quote_text"],
        max_new_tokens=MAX_NEW_TOKENS,
    )

    predictions.append(
        {
            "quote_id": row["quote_id"],
            "gold_category": row["gold_category"],
            "quote_text": row["quote_text"],
            "response": response,
        }
    )

print(len(predictions))

  0%|          | 0/92 [00:00<?, ?it/s]

92


In [ ]:
from utils.inference import parse_response

correct = 0
wrong = 0

for sample in predictions:

    try:

        parsed = parse_response(sample["response"])

        if parsed["category"] == sample["gold_category"]:
            correct += 1
        else:
            wrong += 1

    except Exception as e:

        print("=" * 80)
        print("Failed:", sample["quote_id"])
        print(e)
        print(sample["response"][:1000])
        print("=" * 80)

print("Correct:", correct)
print("Wrong:", wrong)

Failed: Quote 115
Unterminated string starting at: line 6 column 18 (char 634)
[{
"id_quote": "Quote 115",
"related_quote": "So now I want to know what to do. I've spent a lot of my time on this project, so I don't want to just get up and leave, but I'm finding it hard to work with this new developer. [...] On the flip side, he is now the #1 committer on the project, with even more commits than the lead developer. [...] I'm not really sure what to do about this. [...] Has anybody else experienced this problem? If so, what did you do?",
"category": "Expressing concerns",
"challenge_identified": "Uncertainty and frustration regarding adapting to a new teammate and managing expectations.",
"related_quote": "So now I want to know what to do. I've spent a lot of my time on this project, so I don't want to just get up and leave, but I'm finding it hard to work with this new developer. [...] On the flip side, he is now the #1 committer on the project, with even more commits than the lead deve

In [ ]:
# ============================================================
# Build Preference Dataset
# ============================================================

import json

from utils.inference import parse_response
from utils.inference import build_prompt

preferences = []

failed = 0

for sample in predictions:

    try:

        parsed = parse_response(
            sample["response"]
        )

    except Exception:

        failed += 1
        continue

    # Skip correct predictions
    if parsed["category"] == sample["gold_category"]:
        continue

    prompt = build_prompt(
        prompt_template=PROMPT,
        quote_id=sample["quote_id"],
        quote_text=sample["quote_text"],
    )

    chosen = json.dumps(
        [
            {
                "id_quote": sample["quote_id"],
                "category": sample["gold_category"],
                "challenge_identified": parsed.get(
                    "challenge_identified",
                    "",
                ),
                "related_quote": sample["quote_text"],
            }
        ],
        ensure_ascii=False,
    )

    rejected = json.dumps(
        [
            parsed
        ],
        ensure_ascii=False,
    )

    preferences.append(
        {
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected,
        }
    )

print(f"Preference pairs : {len(preferences)}")
print(f"Skipped samples  : {failed}")

Preference pairs : 52
Skipped samples  : 3


In [ ]:
# ============================================================
# Save Preference Dataset
# ============================================================

import json

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as f:

    for sample in preferences:

        json.dump(
            sample,
            f,
            ensure_ascii=False,
        )

        f.write("\n")

print("Preference dataset saved successfully.")
print(OUTPUT_PATH)

Preference dataset saved successfully.
/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM/data/dpo/train.jsonl


In [ ]:
# ============================================================
# Preview Preference Dataset
# ============================================================

import pandas as pd

preview = pd.DataFrame(
    preferences
)

print(preview.shape)

preview.head()

(52, 3)


,prompt,chosen,rejected
0,You are a software engineering researcher spec...,"[{""id_quote"": ""Quote 16"", ""category"": ""Drawing...","[{""id_quote"": ""Quote 16"", ""type"": ""Expressing ..."
1,You are a software engineering researcher spec...,"[{""id_quote"": ""Quote 69"", ""category"": ""Recomme...","[{""id_quote"": ""Quote 69"", ""category"": ""Express..."
2,You are a software engineering researcher spec...,"[{""id_quote"": ""Quote 25"", ""category"": ""Express...","[{""id_quote"": ""Quote 25"", ""category"": ""Drawing..."
3,You are a software engineering researcher spec...,"[{""id_quote"": ""Quote 63"", ""category"": ""Disagre...","[{""id_quote"": ""Quote 63"", ""category"": ""Express..."
4,You are a software engineering researcher spec...,"[{""id_quote"": ""Quote 85"", ""category"": ""Disagre...","[{""id_quote"": ""Quote 85"", ""category"": ""Express..."


In [ ]:
# ============================================================
# Verify Saved Dataset
# ============================================================

with open(
    OUTPUT_PATH,
    encoding="utf-8",
) as f:

    for i in range(3):

        print("=" * 80)
        print(f.readline())

{"prompt": "You are a software engineering researcher specializing in qualitative analysis of psychological safety in software development teams.\n\nYour task is to perform qualitative coding of the quotes provided below by identifying and classifying psychological safety challenges described in each quote.\n\nThe goal is to identify challenges related to psychological safety, such as interpersonal risk-taking, learning anxiety, defensive behaviors, or resistance to change.\n\n---\n\nContext for Classification\n\nPsychological safety refers to a person's perception that they can take interpersonal risks in a team environment without fear of negative consequences such as embarrassment, rejection, or punishment.\n\n---\n\nUnit of Analysis\n\nEach quote represents a single unit of analysis.\n\nEven if multiple psychological challenges appear in the same quote, identify and classify only the ONE primary challenge that best represents the main issue expressed in the quote.\n\n---\n\nInstruc